## 1. Finish EDA

In [ ]:
!pip install langdetect spacy

In [ ]:
# import dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import wordcloud
import seaborn as sns
import langdetect
from langdetect import detect, DetectorFactory
import spacy # Handle english text
import nltk

In [ ]:
#Mount google drive
from google.colab import drive


In [ ]:
#Importing dataset and take a look at the first 5 rows
reddit_employ = pd.read_csv("/content/drive/MyDrive/ML/data/raw/reddit_employment_master.csv")
reddit_employ.head()

In [ ]:
# Review number of entries and columns
reddit_employ.shape

In [ ]:
# Created a copy of the reddit_employ dataset
pd.set_option("display.max_colwidth", None) # Take a peak at the whole text to see the usual pattern of a thread
reddit_df = reddit_employ.copy()
reddit_df.head(3)

Since we intentionally scrape 200 topics from each subreddits, we can expect a pretty balanced distribution among the subreddits

## Data Cleaning and Preprocessing

In [ ]:
# Checking for any null values
reddit_df.isnull().sum()

There're approximately 449 text that are null in the `selftext` column, let's fill null values with blank spaces first for NLP purpose later.

In [ ]:
# Fill `selftext` column with "" first
reddit_df['selftext'] = reddit_df['selftext'].fillna("")
print(reddit_df.isnull().sum())

In [ ]:
# Combine `title` & `selftext` columns together since titles contain the main ideas, selftext contains details => more convenients when doing EDA later
reddit_df['full_text'] = reddit_df['title'] + " " + reddit_df["selftext"]
# check the full_text column
reddit_df['full_text'].head()

Since we intentionally scrape 200 topics from each subreddits, we can expect a pretty balanced distribution among the subreddits

In [ ]:
# Count how many texts are there in each subreddits
subreddit_counts = reddit_df['subreddit'].value_counts().reset_index()
subreddit_counts.columns = ['subreddit', 'count']

plt.figure(figsize =(12, 8))
sns.barplot(data = subreddit_counts,
            x = 'count', y = 'subreddit',
            palette = 'viridis')

plt.tight_layout()
plt.savefig('subreddit_counts.png')
plt.show()

The subreddits are mostly in English, the rest are other languages (including Vietnamese). Since our concerns in this project is about Vietnamese and English, we'll sort out other languages (e.g. French, Japanese, etc.)

## 1. b. Language detection
Since `underthesea` is for Vietnamese, we'll need `Spacy` to work with English

- Devide the datasets into two: `english_df` and `vietnamese_df`, for those text that contains both english and vietnamese (Vinglish), prioritize the one that outnumber

In [ ]:
from langdetect import detect

def get_language(text):
    try:
        return detect(text)
    except:
        return "uknown"

# Apply the function to create another columns
reddit_df['language'] = reddit_df['full_text'].apply(get_language)

# Create 2 different datasets with vietnames and english
reddit_vn = reddit_df[reddit_df['language'] == "vi"].copy()
reddit_en = reddit_df[reddit_df['language'] == "en"].copy()

In [ ]:
# Make sure the results are reproducible
DetectorFactory.seed = 0

lang_counts = reddit_df['language'].value_counts()

# Separate subreddits with english apart from the others
en_counts = lang_counts.get('en', 0)
other_counts = lang_counts.sum() - en_counts
# Create a new series to draw chart
lang_plot_data = pd.Series({
    "English (en)": en_counts,
    "Other languages (vi, fr, ja)": other_counts
})

# Draw a pie chart
plt.figure(figsize = (10, 8))
# colors = ['']
plt.pie(
    lang_plot_data,
    labels=lang_plot_data.index,
    autopct= "%1.1f%%", # show percentage
    startangle=140,
    explode=(0, 0.1),  # Create small spaces between slices for easy visibility
)
plt.title("Distribution of languages in subreddits", fontsize=16, pad=20)
plt.axis('equal')
plt.tight_layout()
plt.savefig("language_distribution_chart.png")
plt.show()

print("\nDetailed Distribution table:")
print(lang_counts)

In [ ]:
print(f"Subreddits contains Vietnamese has {reddit_vn.shape[0]} rows and {reddit_vn.shape[1]} columns")
print(reddit_vn['full_text']) # Take a peek at these subreddits

We can see that with these text, even when there are Vietnamese, some of them are still written mainly in English, we can see that even with immigrants, in this case Vietnamese immigrants, **we still prioritize using English to navigate through the procecss of settling down or receiving helps from international friends**.
=> Therfore, we still use `underthesea` to extract important features from these 7 subreddits but our main focus will be on the rest of the dataset, which is mostly written in English

In [ ]:
def process_vietnamese(text):
  # Tokenize Vietnamese words
  tokens = word_tokenize(text, format="text")
  #
  tags = pos_tag(text)
  return tokens

### English:
Use `Spacy` for lowercase, remove stopwords and Lemmatization all at one.
We don't want to set everything in lowercase first since there might be some "misunderstanding" happening (e.g. python and Python - programming language). Besides, remove punctuation first will affect Spacy ability to parse dependency. For numbers, spacy has IS_DIGIT which will help remove them

In [ ]:
from collections import Counter
# 1. Load model
nlp = spacy.load("en_core_web_sm")
# Define function to process the text
def get_keywords(text):
    doc = nlp(text[:10000])

    # Get PROPN and NOUN
    keywords = [
        token.lemma_.lower() for token in doc # Lemmatization
        if (token.pos_ in ['PROPN', 'NOUN'])
        and not token.is_stop # Remove stopwords
        and not token.is_punct # Remove punctuations
    ]
    return keywords

print ("Extrapolating keywords (might take roughly 1-2 minutes for 2700 texts)...")

# Apply it to the reddit_en
all_keywords = []
for text in reddit_en['full_text']:
    all_keywords.extend(get_keywords(text))

# Get top 20 keywords
top_keywords = Counter(all_keywords).most_common(20)

# Create dataframe to plot the chart
reddit_top_20 = pd.DataFrame(top_keywords, columns = ['Keyword', 'Count'])
print("\nTop 20 keywords:")
print(reddit_top_20)

plt.figure(figsize = (12, 8))
sns.barplot(data = reddit_top_20, x="Count", y="Keyword", palette="magma")
plt.title("Top 20 keywords are debated the most in Reddit")
plt.xlabel("Occurences")
plt.ylabel("Keywords")
plt.savefig('top_20_keywords.png')
plt.show()

From the chart, we can see a few things:
- Top 20 keywords are mostly about **work** (e.g. work, job, experience, position, role, worker, lmia (Labour Market Impact Assessment - very import for immigrants), others words that don't tell much about the topic (e.g. month, day, canada, thing)
- We can see that the immigrants are really concerned about whether they will get LMIA in order to be able to stay in Canada legally for living and working purpose
- If the frequency of LMIA occur with negative words (e.g. scam, fake, pay) increases, it's something that we need to take into account of.

- We can use `Phrase Matching`  to group the similar definitions of immigration/work in Canada:


If we follow the steps above, we will have a list that is, honestly, incomprehensible, we'll use bigrams to take care of this problem (try to match the words in pair - LMIA support, LMIA scam - in order for the sentence to make more sense)

In [ ]:
# Clean some more "Reddit trash"
cleaned_words = [
    word for word in all_keywords if word.isalpha() # Remove !, ?, 123, &
    and len(word) > 2 # Keep words whose len > 2 (remove "i", "am")
    and word not in ['amp', 'http', 'https', 'reddit', 'post',
                     'get', 'know', 'think', 'want'] # Common Reddit trashs
]

In [ ]:
len(cleaned_words)

In [ ]:
from nltk.collocations import BigramAssocMeasures, BigramCollocationFinder
# Instantiate the Bigrams measures
bigram_measures = BigramAssocMeasures()

# Create a finder
finder = BigramCollocationFinder.from_words(cleaned_words)

# Only keep the words appearing > 3 times to remove trash
finder.apply_freq_filter(3)

# Filter: only keeps the words that have "lmia"
lmia_filter = lambda w1, w2: "lmia" not in (w1.lower(), w2.lower())
finder.apply_ngram_filter(lmia_filter)

# 3. Tot 10 most popular "lmia terms"
top_lmia_phrases = finder.nbest(bigram_measures.pmi, 10)
print("Top 10 most meaningful terms related to LMIA")
for phrase in top_lmia_phrases:
  print(f"{phrase[0]} {phrase[1]}")


In [ ]:
# Plot a horizontal bar chart about top 10 LMIA keywords
plt.figure(figsize = (12, 8))
lmia_data = pd.DataFrame(top_lmia_phrases, columns = ['Phrases', 'Frequency'])
sns.set_style("whitegrid")
ax = sns.barplot(data = lmia_data,
            x = 'Frequency', y = 'Phrases',
            palette = 'Blues_d')

# Adding numbers for better visibility
for i in ax.containers:
  ax.bar_label(i,padding=3)

plt.title("Top 10 most common phrases discussing about LMIA", fontsize=15, fontweight="bold")
plt.xlabel("Frequency", fontsize=12)
plt.ylabel("Phrases", fontsize=12)
plt.tight_layout()
plt.show()

**Time Series Line chart (post Volume/Time)**: Theo dõi số lượng bài đăng theo tuần/tháng. Nếu có 1 đợt sóng bài đăng đột ngột, có thể do 1 chính sách mới vừa ra đời.

**Stacked Bar Chart (Top skills mentioned by Industry)**: phân loại được ngành nghề (IT, Marketing, ...) biểu đồ này sẽ chỉ ra SKILLS nào đang chiếm sóng trong mỗi tuần

### Plotting weekly trends
1. post per subreddits
2. posts containing city names
3. posts mentioning ("visa sponsor", "remote", "warehouse", "nanny", "cash jobs")
4. posts flagged as suspicious (regex-based)
=> This will be the foundation for dashboard

Plot for hard skills likes: Python, SQL, Java, etc.
Plot for keywords related to "LMIA, PR, Permit"

In [1]:
# Extra plots for programming languages that are being discussed more recently

=> create insights:
- Which city is the most popular
- How are job categories distributed according to subreddit?

## Sentimental Analysis ML model!
- Purpose: understand the public's emotions based on real-time trends that are expressed through words in Reddits

### a. Initial Sentiment EDA
- Use `TextBlob` library (for English) to see general attitude (negative or positive) before diving deeper into more complex models
- we can see **Polarity** (-1 -> 1) and **Subjectivity** (0 -> 1) - the higher the number, the more objective it's, the lower, the more truth it bears

In [ ]:
from typing import Text
from textblob import TextBlob
def get_sentiment_score(text):
  return TextBlob(text).sentiment.polarity

# create a collumn named "sentiment_score" for reddit_en
reddit_en['sentiment_score'] = reddit_en['full_text'].apply(get_sentiment_score)

# Function to compare emotions between keywords
def analyze_keyword_sentiment(keyword, dataframe):
  # Filter text with keywords, either upper or lowercase
  relevant_posts = dataframe[dataframe['full_text'].str.contains(keyword, case=False)]
  avg_score = relevant_posts['sentiment_score'].mean()
  count = len(relevant_posts)
  return avg_score, count

# Test with LMIA and programming languages
keywords_test = ['lmia', 'python', 'java', 'sql', 'javascript']
results = []
for k in keywords_test:
  score, count = analyze_keyword_sentiment(k, reddit_en)
  results.append({"Keyword": k, "Avg Sentiment": score, "Post count": count})

df_sentiment = pd.DataFrame(results)
print(df_sentiment)

In [ ]:
# Let see how emotions are tie to these keywords
plt.figure(figsize=(10, 6))
sns.barplot(data=df_sentiment, x='Keyword', y='Avg Sentiment', palette='RdYlGn')
plt.axhline(0, color="black", linestyle="--", linewidth = 1)
plt.title("Comparing emotions between LMIA and programming languages")
plt.ylabel("Average Sentiment score (-1 to 1)")
plt.xlabel("Keywords")
plt.tight_layout()
plt.savefig("sentiment_analysis.png")
plt.show()

From this chart, we can see that there's a clear polarity in the immigrants' minds.
* When it comes to LMIA, this topic is the most stressful and concerning with the immigrants since it's heavily tied to the chance of being able to settle down and working in Canada.
- Users usually ask questions such as (How to..., I need help with..., Can someone explain...) and they usually include words like "please, thanks" that inadvertently increase the score a bit.
* While the programming skills bears more positivity, usually these skills are included in the subreddits related to **Web Development, React, FreeLance** these are the constructive topics, with less or nothing having to do with emotions.
* Regarding *Java*, this keyword usually appear in subreddits about Enterprise, which might explain why the emotional score is a bit lower (stable job with less excitement).

=> Insight for the recruiters: Immigrants workers are especially sensitive and a bit guarding against anything that is related to LMIA, company and enterprise should be more clear when it comes to this matter to build trust.

In [ ]:
# Get posts that are related to Python keywors
python_posts = reddit_en[reddit_en['full_text'].str.contains(r'\bpython\b', case=False, na=False, regex=True)].copy()
python_posts.shape

In [ ]:
pet_keywords = ['snake', 'ball', 'pet', 'animal',
                'morph', 'tank', 'rehome', 'breed', 'reptile']
def check_python_context(text):
  if any(word in text for word in pet_keywords):
    return "Pet/Animal"
  return "Coding/Tech"

# Take a peek at the categories after processing
python_posts['category'] = python_posts['full_text'].apply(check_python_context)


We can see that even with processing and trying to separate the Python programming skill apart from python the real animal, there're still some mismatch in categorizing, however this also implies that **most of the subreddits we scraped are truly from people concerning about their jobs and the job markets, which fortifies our finding above, that Python, SQL and javascript are still one of the programming languages that most people are learning.**

In [ ]:
# Get some examples about LMIA
scam_keywords = ['scam', 'fake', 'loophole', 'fraud', 'illegal', 'pay']
lm_scam_pattern = "|".join(scam_keywords)
lm_scam_posts = reddit_en[reddit_en['full_text'].str.contains('lmia', case=False) &
                  reddit_en['full_text'].str.contains(lm_scam_pattern, case=False)
]

print(f"Found {len(lm_scam_posts)} subreddits that might related to LMIA Scams")

In [ ]:
print("\n-- Some examples about LMIA scams for recruiters--")
for i, row in lm_scam_posts.head(3).iterrows():
  print(f"Subreddit: {row['subreddit']}")
  print(f"Title: {row['title']}")
  print(f"{TextBlob(row['full_text']).sentiment.polarity:.2f}")

1.3. Topic Modelling
- LDA
- BERTopic (much better). From this, we'll see
=> Tops popular topic weekly
- Which topic might seems like SCAM?
- Which topic is hot in Canada

## 2. SCAM detection ML model!
- Purpose: check if a post is categoriez as SCAM or not based on learned information

1.   Baseline model
- TF-IDF + Logistic regression
- TF-IDF + Linear SVM
- Naive Bayes
=> Give us a good baseline model
2.   After that, upgrade to strong model
- DistilBERT + fine-tuning
- RoBERTa-base -> very strong for classification
=> these ones can catch deeper scam patterns (tone, phrasing)

### Some keywords Analysis for Scam detectors
Create binary features:
- contains_whatsapp
- contains_telegram
- contains_fee
- contains_cash_job
- contains_no_company_name
- contains_visa_sponsorship
These are features for the model.



1.2. Extract structured signals from posts
**Generate additional features** such as:
- Location extractions (Canada cities -> NEX or Regex)
- Salary mentions extraction ($xx/hr)
- Contract type (full-time, part-time, internship...)
- Recruiters keywords ("DM me", "message me on Whatsapp", "telegram", "upfront fee"
=> these features might be critical to **detect scam**

Create a CSV for Labelling
Columns:
- post_id
- title
- body
- subreddit
- scam_label (initially empty)
Export it to simple Google Sheet

2.1. Label ~300-500 posts (cannot train without labels)
- scam = 1
- legit = 0

Signs for scam can be:
- WhatsApp contact
- Telegram link
- Upfront payment
- Too-good-to-be-true salary
- No company name
- Links to unfamiliar domains
- Referral fee
- Duplicate posts across subreddits
- Poor English + inconsistent formatting

2.2. Choose model






2.3. Evaluation
- Precision: most important - we don't want to flag thea legit posts
- Recall
- ROC-AUC

## 3. Build interactive dashboard
3.1. Scraper schedule
- Runs automatically every monday
- Use GitHub Actions or Python Cron
- Scrape every subreddit
- Save to S3 / local/ SQLite/ PostgreSQL

3.2. Dashboard features.

Use Streamlit or Dash. Should have
- Live subreddits statistics.
- Job trends by city
- Salary mentions trend
- Keyword monitoring ("visa sponsor", "LMIA", "cash job")
- Scam detector, (highlight suspicious posts)
- Filter by time, location, subreddit

3.3. Live inference

When scraping subreddits:
- Clean text
- Input into model
- Tag SCAM probabilties
- Save to database
- Show in dashboard